In [0]:
dbutils.library.restartPython()

In [0]:
import os, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import common, jobs

# jobs/01_deploy_jobs: create or update this repo's daily job from config/jobs.yml.
# dev: run by the Developer (job runs as the Developer, on demand).
# test/prod: run by the admin login (job runs as the environment's job account, schedule paused until go-live).
dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
dbutils.widgets.dropdown("mode", "plan", ["plan", "apply"])
env, mode = dbutils.widgets.get("env"), dbutils.widgets.get("mode")
cfg = common.load_config(env)
root = os.path.dirname(os.getcwd())
r = jobs.deploy(jobs.Api(WorkspaceClient()), jobs.load_jobs(root), root, cfg["code"], env, apply=(mode == "apply"))
print(f"mode={mode}")
for k in ("job", "action", "job_id", "run_as", "schedule", "tasks"):
    print(f"{k:9} {r[k]}")
if mode == "apply":
    common.log_run(spark, cfg, "jobs_deploy", "success", 1, f"{r['job']} {r['action']}")